Imports

In [17]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

Load Customer Features

In [18]:
user_features = pd.read_parquet(
    "processed/user_features.parquet"
)

print("Shape:", user_features.shape)
print("\nColumns:")
print(user_features.columns.tolist())
user_ids = user_features["user_id"].to_numpy()

feature_columns = [
    column
    for column in user_features.columns
    if column != "user_id"
]

X = user_features[
    feature_columns
].to_numpy(dtype=float)

print("\nNumber of users:", X.shape[0])
print("Number of features:", X.shape[1])
print("Missing values:", np.isnan(X).sum())
print("Infinite values:", np.isinf(X).sum())

Shape: (206209, 21)

Columns:
['user_id', 'alcohol', 'babies', 'bakery', 'beverages', 'breakfast', 'bulk', 'canned_goods', 'dairy_eggs', 'deli', 'dry_goods_pasta', 'frozen', 'household', 'international', 'meat_seafood', 'other', 'pantry', 'personal_care', 'pets', 'produce', 'snacks']

Number of users: 206209
Number of features: 20
Missing values: 0
Infinite values: 0


Map Reduce Parameters (including shuffling)

In [19]:
K = 4

CHUNK_SIZE = 20000

RANDOM_STATE = 42

MAX_ITERATIONS = 100

TOLERANCE = 1e-6

In [20]:
rng = np.random.default_rng(
    RANDOM_STATE
)

shuffle_indices = rng.permutation(
    len(X)
)

X_shuffled = X[
    shuffle_indices
]

user_ids_shuffled = user_ids[
    shuffle_indices
]

print("Shuffled X shape:", X_shuffled.shape)

Shuffled X shape: (206209, 20)


K Means Initialization

In [21]:
def farthest_first_initialization(
    X,
    k,
    random_state=42
):
    rng = np.random.default_rng(
        random_state
    )

    first_index = rng.integers(
        0,
        X.shape[0]
    )

    centroid_indices = [
        first_index
    ]

    centroids = [
        X[first_index].copy()
    ]

    while len(centroids) < k:

        current_centroids = np.array(
            centroids
        )

        distances = np.linalg.norm(
            X[:, np.newaxis, :]
            - current_centroids[
                np.newaxis, :, :
            ],
            axis=2
        )

        min_distances = distances.min(
            axis=1
        )

        min_distances[
            centroid_indices
        ] = -1

        next_index = np.argmax(
            min_distances
        )

        centroid_indices.append(
            next_index
        )

        centroids.append(
            X[next_index].copy()
        )

    return np.array(
        centroids
    )

In [22]:
# Initialize the K global centroids once before MapReduce begins

global_centroids = farthest_first_initialization(
    X_shuffled,
    k=K,
    random_state=RANDOM_STATE
)

print("Initial global centroid shape:", global_centroids.shape)

Initial global centroid shape: (4, 20)


Each Mapper needs to cluster its own chunk

In [23]:
def local_kmeans(
    X,
    k,
    random_state=42,
    max_iterations=100,
    tolerance=1e-6
):
    centroids = farthest_first_initialization(
        X,
        k=k,
        random_state=random_state
    )

    for iteration in range(
        max_iterations
    ):
        distances = np.linalg.norm(
            X[:, np.newaxis, :]
            - centroids[
                np.newaxis, :, :
            ],
            axis=2
        )

        labels = np.argmin(
            distances,
            axis=1
        )

        new_centroids = (
            centroids.copy()
        )

        for cluster_id in range(k):

            cluster_points = X[
                labels == cluster_id
            ]

            if len(cluster_points) > 0:

                new_centroids[
                    cluster_id
                ] = cluster_points.mean(
                    axis=0
                )

        centroid_shift = np.linalg.norm(
            new_centroids
            - centroids
        )

        centroids = new_centroids

        if centroid_shift < tolerance:
            break

    # Final assignment
    distances = np.linalg.norm(
        X[:, np.newaxis, :]
        - centroids[
            np.newaxis, :, :
        ],
        axis=2
    )

    labels = np.argmin(
        distances,
        axis=1
    )

    return (
        labels,
        centroids,
        iteration + 1
    )

Cluster Description (N, Sum, Sumq, centroid)

In [24]:
def summarize_cluster(points):

    points = np.asarray(
        points,
        dtype=float
    )

    N = len(points)

    SUM = np.sum(
        points,
        axis=0
    )

    SUMSQ = np.sum(
        points ** 2,
        axis=0
    )

    centroid = SUM / N

    return {
        "N": N,
        "SUM": SUM,
        "SUMSQ": SUMSQ,
        "centroid": centroid
    }

Map (receives: mapper_id, chuunk) and (emits: key, value)

In [ ]:
pairs, stats = map_task(
    mapper_id=mapper_id,
    chunk=chunk,
    global_centroids=global_centroids
):
    start_time = time.perf_counter()

    distances = np.linalg.norm(
        chunk[:, np.newaxis, :]
        - global_centroids[np.newaxis, :, :],
        axis=2
    )

    labels = np.argmin(
        distances,
        axis=1
    )

    intermediate_pairs = []

    for cluster_id in range(
        len(global_centroids)
    ):
        cluster_points = chunk[
            labels == cluster_id
        ]

        if len(cluster_points) == 0:
            continue

        value = {
            "mapper_id": mapper_id,
            "N": len(cluster_points),
            "SUM": cluster_points.sum(axis=0),
            "SUMSQ": (
                cluster_points ** 2
            ).sum(axis=0)
        }

        # Intermediate key is the global cluster ID
        intermediate_pairs.append(
            (cluster_id, value)
        )

    runtime = (
        time.perf_counter()
        - start_time
    )

    stats = {
        "mapper_id": mapper_id,
        "input_points": len(chunk),
        "output_pairs": len(
            intermediate_pairs
        ),
        "runtime": runtime
    }

    return intermediate_pairs, stats

Test One Map Task first

In [26]:
test_chunk = X_shuffled[
    :CHUNK_SIZE
]

test_pairs, test_stats = map_task(
    mapper_id=0,
    chunk=test_chunk,
    k=K
)

print("Map test")
print("--------")

print(
    "Input points:",
    test_stats["input_points"]
)

print(
    "Intermediate pairs:",
    test_stats["output_pairs"]
)

print(
    "Iterations:",
    test_stats["iterations"]
)

print(
    f"Runtime: "
    f"{test_stats['runtime']:.4f}s"
)
for key, value in test_pairs:

    print(
        f"key={key}, "
        f"mapper={value['mapper_id']}, "
        f"local cluster={value['local_cluster_id']}, "
        f"N={value['N']}"
    )

mapped_points = sum(
    value["N"]
    for key, value in test_pairs
)

print(
    "Points represented by Map output:",
    mapped_points
)

assert mapped_points == len(
    test_chunk
)

TypeError: map_task() got an unexpected keyword argument 'k'

Complete Map Stage

In [ ]:
all_intermediate_pairs = []

map_history = []

map_stage_start = (
    time.perf_counter()
)

mapper_id = 0

for start in range(
    0,
    len(X_shuffled),
    CHUNK_SIZE
):
    end = min(
        start + CHUNK_SIZE,
        len(X_shuffled)
    )

    chunk = X_shuffled[
        start:end
    ]

    pairs, stats = map_task(
        mapper_id=mapper_id,
        chunk=chunk,
        k=K
    )

    all_intermediate_pairs.extend(
        pairs
    )

    map_history.append(
        stats
    )

    print(
        f"Mapper {mapper_id}: "
        f"{stats['input_points']:,} input points, "
        f"{stats['output_pairs']} emitted pairs, "
        f"{stats['iterations']} iterations, "
        f"{stats['runtime']:.4f}s"
    )

    mapper_id += 1

map_stage_runtime = (
    time.perf_counter()
    - map_stage_start
)

print(
    "\nTotal Map-stage runtime:",
    f"{map_stage_runtime:.4f}s"
)

Validate Map Stage

In [ ]:
total_mapped_points = sum(
    value["N"]
    for key, value
    in all_intermediate_pairs
)

print(
    "Original customer points:",
    f"{len(X_shuffled):,}"
)

print(
    "Points represented by "
    "intermediate summaries:",
    f"{total_mapped_points:,}"
)

print(
    "Intermediate key-value pairs:",
    len(all_intermediate_pairs)
)

assert (
    total_mapped_points
    == len(X_shuffled)
)

Explicit shuffle / group

In [ ]:
def shuffle_and_group(
    intermediate_pairs
):
    grouped = {}

    for key, value in intermediate_pairs:

        if key not in grouped:
            grouped[key] = []

        grouped[key].append(value)

    return grouped

Reduce Stage

In [ ]:
def weighted_farthest_first(
    centroids,
    weights,
    k,
    random_state=42
):
    rng = np.random.default_rng(
        random_state
    )

    first_index = rng.integers(
        0,
        len(centroids)
    )

    selected_indices = [
        first_index
    ]

    global_centroids = [
        centroids[first_index].copy()
    ]

    while len(global_centroids) < k:

        current_centroids = np.array(
            global_centroids
        )

        distances = np.linalg.norm(
            centroids[:, np.newaxis, :]
            - current_centroids[
                np.newaxis, :, :
            ],
            axis=2
        )

        min_distances = distances.min(
            axis=1
        )

        min_distances[
            selected_indices
        ] = -1

        next_index = np.argmax(
            min_distances
        )

        selected_indices.append(
            next_index
        )

        global_centroids.append(
            centroids[next_index].copy()
        )

    return np.array(
        global_centroids
    )

Weighted Clustering of Map Summaries

In [ ]:
def cluster_map_summaries(
    descriptions,
    k,
    random_state=42,
    max_iterations=100,
    tolerance=1e-6
):
    local_centroids = np.vstack([
        description["centroid"]
        for description in descriptions
    ])

    weights = np.array([
        description["N"]
        for description in descriptions
    ], dtype=float)


    for iteration in range(
        max_iterations
    ):

        distances = np.linalg.norm(
            local_centroids[
                :, np.newaxis, :
            ]
            - global_centroids[
                np.newaxis, :, :
            ],
            axis=2
        )

        assignments = np.argmin(
            distances,
            axis=1
        )

        new_centroids = (
            global_centroids.copy()
        )

        for cluster_id in range(k):

            mask = (
                assignments == cluster_id
            )

            if np.any(mask):

                new_centroids[
                    cluster_id
                ] = np.average(
                    local_centroids[mask],
                    axis=0,
                    weights=weights[mask]
                )

        centroid_shift = np.linalg.norm(
            new_centroids
            - global_centroids
        )

        global_centroids = (
            new_centroids
        )

        if centroid_shift < tolerance:
            break

    # Final assignments
    distances = np.linalg.norm(
        local_centroids[:, np.newaxis, :]
        - global_centroids[
            np.newaxis, :, :
        ],
        axis=2
    )

    assignments = np.argmin(
        distances,
        axis=1
    )

    return (
        assignments,
        global_centroids,
        iteration + 1
    )

Merge N, Sum, Sumq

In [ ]:
def merge_cluster_descriptions(
    descriptions,
    assignments,
    k
):
    global_summaries = {}

    for cluster_id in range(k):

        mask_indices = np.where(
            assignments == cluster_id
        )[0]

        if len(mask_indices) == 0:
            continue

        cluster_descriptions = [
            descriptions[index]
            for index in mask_indices
        ]

        total_N = sum(
            description["N"]
            for description
            in cluster_descriptions
        )

        total_SUM = np.sum(
            [
                description["SUM"]
                for description
                in cluster_descriptions
            ],
            axis=0
        )

        total_SUMSQ = np.sum(
            [
                description["SUMSQ"]
                for description
                in cluster_descriptions
            ],
            axis=0
        )

        global_summaries[
            cluster_id
        ] = {
            "N": total_N,
            "SUM": total_SUM,
            "SUMSQ": total_SUMSQ,
            "centroid": (
                total_SUM / total_N
            )
        }

    return global_summaries

Define Reduce()

In [ ]:
def reduce_task(
    cluster_id,
    summaries
):
    total_N = sum(
        summary["N"]
        for summary in summaries
    )

    total_SUM = np.sum(
        [
            summary["SUM"]
            for summary in summaries
        ],
        axis=0
    )

    total_SUMSQ = np.sum(
        [
            summary["SUMSQ"]
            for summary in summaries
        ],
        axis=0
    )

    centroid = (
        total_SUM / total_N
    )

    return {
        "cluster_id": cluster_id,
        "N": total_N,
        "SUM": total_SUM,
        "SUMSQ": total_SUMSQ,
        "centroid": centroid
    }

In [ ]:
def run_mapreduce_iteration(
    X,
    global_centroids,
    chunk_size
):
    all_intermediate_pairs = []
    map_history = []

    map_start = time.perf_counter()

    mapper_id = 0

    for start in range(
        0,
        len(X),
        chunk_size
    ):
        end = min(
            start + chunk_size,
            len(X)
        )

        chunk = X[start:end]

        pairs, mapper_stats = map_task(
            mapper_id=mapper_id,
            chunk=chunk,
            global_centroids=global_centroids
        )

        all_intermediate_pairs.extend(
            pairs
        )

        map_history.append(
            mapper_stats
        )

        mapper_id += 1

    map_runtime = (
        time.perf_counter()
        - map_start
    )

    shuffle_start = time.perf_counter()

    grouped = shuffle_and_group(
        all_intermediate_pairs
    )

    shuffle_runtime = (
        time.perf_counter()
        - shuffle_start
    )

    reduce_start = time.perf_counter()

    reduced_clusters = {}

    for cluster_id, summaries in grouped.items():

        reduced_clusters[
            cluster_id
        ] = reduce_task(
            cluster_id,
            summaries
        )

    reduce_runtime = (
        time.perf_counter()
        - reduce_start
    )

    new_centroids = np.vstack([
        reduced_clusters[
            cluster_id
        ]["centroid"]
        for cluster_id in range(K)
    ])

    stats = {
        "map_runtime": map_runtime,
        "shuffle_runtime": shuffle_runtime,
        "reduce_runtime": reduce_runtime,
        "map_tasks": len(map_history),
        "intermediate_pairs": len(
            all_intermediate_pairs
        )
    }

    return (
        new_centroids,
        reduced_clusters,
        map_history,
        stats
    )


Run Reduce()

In [ ]:
mapreduce_history = []

global_centroids = (
    farthest_first_initialization(
        X_shuffled,
        k=K,
        random_state=RANDOM_STATE
    )
)

final_clusters = None

overall_start = time.perf_counter()

for iteration in range(
    MAX_ITERATIONS
):
    (
        new_centroids,
        reduced_clusters,
        mapper_stats,
        iteration_stats
    ) = run_mapreduce_iteration(
        X=X_shuffled,
        global_centroids=global_centroids,
        chunk_size=CHUNK_SIZE
    )

    centroid_shift = np.linalg.norm(
        new_centroids
        - global_centroids
    )

    iteration_stats["iteration"] = (
        iteration + 1
    )

    iteration_stats["centroid_shift"] = (
        centroid_shift
    )

    mapreduce_history.append(
        iteration_stats
    )

    global_centroids = new_centroids
    final_clusters = reduced_clusters

    print(
        f"Iteration {iteration + 1}: "
        f"shift={centroid_shift:.8f}, "
        f"map={iteration_stats['map_runtime']:.4f}s, "
        f"reduce={iteration_stats['reduce_runtime']:.6f}s"
    )

    if centroid_shift < TOLERANCE:
        print("MapReduce K-means converged.")
        break

total_mapreduce_runtime = (
    time.perf_counter()
    - overall_start
)

In [ ]:
print("Reducer keys:", reduce_results.keys())

In [ ]:
final_clusters = reduce_results[1]

print("Final clusters:", len(final_clusters))

Validate no customers were lost

In [ ]:
print("\nFinal MapReduce Cluster Sizes")
print("-----------------------------")

total_points = 0

for cluster_id in range(K):

    cluster_size = (
        final_clusters[
            cluster_id
        ]["N"]
    )

    total_points += cluster_size

    print(
        f"Cluster {cluster_id}: "
        f"{cluster_size:,} users"
    )

print(
    "\nTotal represented:",
    f"{total_points:,}"
)

print(
    "Expected:",
    f"{len(X_shuffled):,}"
)

assert total_points == len(X_shuffled)
assert len(final_clusters) == K

Build Final Centroid Dataframe

In [ ]:
mapreduce_centroid_df = pd.DataFrame(
    global_centroids,
    columns=feature_columns
)

mapreduce_centroid_df.index.name = (
    "cluster"
)

mapreduce_centroid_df

for cluster_id in range(K):

    print(
        f"\nMapReduce Cluster "
        f"{cluster_id}"
    )

    print("--------------------")

    print(
        mapreduce_centroid_df
        .loc[cluster_id]
        .sort_values(
            ascending=False
        )
        .head(5)
    )

Inspect customer segments

In [ ]:
for cluster_id in (
    mapreduce_centroid_df.index
):

    print(
        f"\nMapReduce Cluster "
        f"{cluster_id}"
    )

    print("--------------------")

    print(
        mapreduce_centroid_df
        .loc[cluster_id]
        .sort_values(
            ascending=False
        )
        .head(5)
    )

Final Cluster sizes

In [ ]:
print("Final MapReduce Cluster Sizes")
print("-----------------------------")

for cluster_id in sorted(
    final_clusters.keys()
):

    print(
        f"Cluster {cluster_id}: "
        f"{final_clusters[cluster_id]['N']:,} "
        f"users"
    )


End to End Run Time

In [ ]:
total_mapreduce_runtime = (
    map_stage_runtime
    + shuffle_runtime
    + reduce_stage_runtime
)

print("MapReduce Runtime Summary")
print("-------------------------")

print(
    f"Map stage: "
    f"{map_stage_runtime:.4f}s"
)

print(
    f"Shuffle/group: "
    f"{shuffle_runtime:.6f}s"
)

print(
    f"Reduce stage: "
    f"{reduce_stage_runtime:.4f}s"
)

print(
    f"Total measured runtime: "
    f"{total_mapreduce_runtime:.4f}s"
)

Map Task Statistics

In [ ]:
map_history_df = pd.DataFrame(
    map_history
)

map_history_df

print(
    "Number of Map tasks:",
    len(map_history_df)
)

print(
    "Average mapper runtime:",
    map_history_df[
        "runtime"
    ].mean()
)

print(
    "Slowest mapper runtime:",
    map_history_df[
        "runtime"
    ].max()
)

print(
    "Fastest mapper runtime:",
    map_history_df[
        "runtime"
    ].min()
)


Summary/Shuffle Volume

In [ ]:
D = X.shape[1]

summary_values_per_cluster = (
    2 * D + 1
)

number_of_descriptions = len(
    all_intermediate_pairs
)

intermediate_numeric_values = (
    number_of_descriptions
    * summary_values_per_cluster
)

original_numeric_values = (
    X_shuffled.shape[0]
    * X_shuffled.shape[1]
)

compression_ratio = (
    intermediate_numeric_values
    / original_numeric_values
)

print(
    "Original feature values:",
    f"{original_numeric_values:,}"
)

print(
    "Intermediate summary values:",
    f"{intermediate_numeric_values:,}"
)

print(
    "Intermediate/original ratio:",
    f"{compression_ratio:.6f}"
)

print(
    "Approximate reduction:",
    f"{(1 - compression_ratio) * 100:.2f}%"
)